<a href="https://colab.research.google.com/github/sachinn854/Brain-Tumor-Segmentatiton/blob/main/notebooks/smoke_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WAS-Mamba — Colab Smoke Test

Purpose: confirm the base-model code actually runs on a real GPU (dependencies install, architecture builds, a forward pass completes) — **no BraTS data needed for this notebook.**

Before running: **Runtime -> Change runtime type -> T4 GPU**, then Runtime -> Run all.

## 1. Get the code

If the repo is **public**, leave `USE_TOKEN = False` below and just run the cell.
If it's **private**, set `USE_TOKEN = True` — it'll prompt for a GitHub personal access token (github.com -> Settings -> Developer settings -> Personal access tokens, `repo` scope), not saved anywhere.

(Only one clone path runs — using "Run all" here is safe and won't create a nested duplicate folder.)

In [1]:
!git -C /content/Brain-Tumor-Segmentatiton pull

Already up to date.


In [2]:
import os

USE_TOKEN = False  # set True for a private repo

REPO_URL = "github.com/sachinn854/Brain-Tumor-Segmentatiton.git"
REPO_DIR = "Brain-Tumor-Segmentatiton"

if not os.path.isdir(f"/content/{REPO_DIR}"):
    if USE_TOKEN:
        import getpass
        token = getpass.getpass('GitHub personal access token: ')
        get_ipython().system(f'git clone https://{token}@{REPO_URL}')
        del token
    else:
        get_ipython().system(f'git clone https://{REPO_URL}')
else:
    print(f"{REPO_DIR} already exists, skipping clone (delete the folder first if you want a fresh clone).")

get_ipython().run_line_magic('cd', f'/content/{REPO_DIR}')

Brain-Tumor-Segmentatiton already exists, skipping clone (delete the folder first if you want a fresh clone).
/content/Brain-Tumor-Segmentatiton


## 2. Check GPU

In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 3. Install dependencies

`torch` is already on Colab. `causal-conv1d` and `mamba-ssm` compile CUDA code at install time.

Two things make this build fast instead of taking 20-40 minutes:
- **`--no-build-isolation`**: without it, pip builds these in a clean throwaway environment that can't see the `torch` already installed on Colab, and the build fails outright with an opaque wheel-build error.
- **`TORCH_CUDA_ARCH_LIST`**: by default these packages compile CUDA code for ~10 different GPU architectures "just in case". Colab's GPU is always one specific one, so this pins the build to only that one — the single biggest speedup here. `nvidia-smi` above already ran, so this cell reads the actual GPU's compute capability instead of guessing.

In [4]:
import os
import torch

# Pin the CUDA build to exactly this GPU's compute capability instead of
# compiling for ~10 architectures -- this alone typically turns a
# 20-40 minute build into a 3-5 minute one.
major, minor = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
print(f"Building only for compute capability {major}.{minor}")

!pip install -q einops timm nibabel ninja packaging
!pip install -q causal-conv1d --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation

Building only for compute capability 7.5


If the cell above still fails to build: **Runtime -> Restart session**, then run every cell from the top again once — a partially-failed build can leave stale files that break a retry in the same session. If it fails a second time with a clean restart, the actual cause is usually in the first ~20 lines of the error (the real compiler error), not the pip wrapper text at the bottom — that's the part worth sharing to debug further.

In [5]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

from src.models.wasmamba import WASMamba
print('Model imported OK')

CUDA available: True
Device: Tesla T4


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Model imported OK


## 5. Build the model with BraTS-shaped settings

`input_channels=4` (FLAIR, T1w, T1Gd, T2w), `num_classes=4` (background + 3 tumor classes) -- matches `src/configs/wasmamba_config.py`.

In [6]:
model = WASMamba(
    input_channels=4,
    num_classes=4,
    depths=[1, 1, 1, 1],
    depths_decoder=[1, 1, 1, 1],
    drop_path_rate=0.2,
).cuda()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model built OK -- {n_params/1e6:.2f}M parameters')

Model built OK -- 24.95M parameters


## 6. Forward pass on a small dummy volume

Using a small 32^3 crop here (not the paper's 128^3) on purpose -- this cell is only checking "does the code run without crashing", not fitting the real training config. The 128^3 / batch=2 config is what actually goes in `src/configs/wasmamba_config.py` for real training.

In [7]:
dummy_input = torch.randn(1, 4, 128, 128, 128).cuda()

with torch.no_grad():
    output = model(dummy_input)

print('Input shape: ', dummy_input.shape)
print('Output shape:', output.shape)
print('Forward pass OK -- architecture runs end-to-end on GPU')

/content/Brain-Tumor-Segmentatiton/src/models/wasmamba.py:475: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at /pytorch/aten/src/ATen/native/Copy.cpp:308.)
  freq = torch.fft.fftn(x, dim=(2, 3, 4)).to(torch.float32)


Input shape:  torch.Size([1, 4, 128, 128, 128])
Output shape: torch.Size([1, 4, 128, 128, 128])
Forward pass OK -- architecture runs end-to-end on GPU


## 7. Sanity-check the loss function too

In [8]:
from src.losses.losses import PaperDiceCeLoss

criterion = PaperDiceCeLoss(num_classes=4)
dummy_label = torch.randint(0, 4, (1, 128, 128, 128)).cuda()

loss = criterion(output, dummy_label)
print('Loss value:', loss.item())
print('Loss function OK')

Loss value: 2.133396625518799
Loss function OK


---
If every cell above ran without error: the base model, its loss, and the GPU environment are all confirmed working. Next step is plugging in real BraTS data (`src/data/brats_dataset.py`) once it's downloaded and on Drive -- not part of this notebook.

## 8. Download a small BraTS2021 test subset

Goal here: confirm `src/data/brats_dataset.py` can actually load a real BraTS case end-to-end -- **not** a full download, not training. Just a mechanism test.

**Before running:** add your Kaggle API token as a Colab Secret first (left sidebar -> key icon -> "Add new secret" -> name `KAGGLE_API_TOKEN`, paste the token as the value, turn on notebook access). Never paste the raw token into a cell -- Colab Secrets keeps it out of the notebook file entirely.

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print("Kaggle token loaded from Colab Secrets (not written to any file)")

!pip install -q kaggle

First, just **list** what's in the dataset (no download yet) -- this confirms the exact file/folder naming on this particular Kaggle mirror before pulling anything, so we download the right few files instead of guessing.

In [ ]:
!kaggle datasets files -d dschettler8845/brats-2021-task1 | head -30

Good news from the listing: individual cases are separate small `.tar` files (~10MB each) alongside the full 13.4GB archive. So we only need to download **one** ~10MB file, not the full dataset.

In [ ]:
!kaggle datasets download -d dschettler8845/brats-2021-task1 -f BraTS2021_00495.tar
!mkdir -p /content/brats_test_case
!tar -xf BraTS2021_00495.tar -C /content/brats_test_case
!find /content/brats_test_case -maxdepth 3

Run the cell above, then paste back the `find` output — it shows the extracted folder/file names, which need to match `src/data/brats_dataset.py`'s expected naming (`_flair.nii.gz`, `_t1.nii.gz`, `_t1ce.nii.gz`, `_t2.nii.gz`, `_seg.nii.gz`) before we can test loading this case through `BratsDataset`.